In [3]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, Flatten,
    Conv1D, MaxPooling1D,
    LSTM, Input, Concatenate
)
from tensorflow.keras.utils import to_categorical


In [4]:
df = pd.read_csv("/content/epilepsy_2.csv")

df.head()


FileNotFoundError: [Errno 2] No such file or directory: '/content/epilepsy_2.csv'

In [ ]:
print("Dataset Shape:", df.shape)
print("\nClass Distribution:\n", df["target_class"].value_counts())


In [ ]:
X = df.drop("target_class", axis=1)
y = df["target_class"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
X_train_seq = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_seq  = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

print("Sequence Shape:", X_train_seq.shape)


In [ ]:
y_train_cat = to_categorical(y_train, num_classes=4)
y_test_cat  = to_categorical(y_test, num_classes=4)


In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix of Features")
plt.show()


In [ ]:
df["target_class"].value_counts().plot(kind="bar")

plt.title("Class Distribution")
plt.xlabel("Condition Class")
plt.ylabel("Samples")
plt.show()


In [ ]:
df.groupby("target_class")["heart_rate"].mean().plot(kind="line", marker="o")

plt.title("Heart Rate Pattern Across Classes")
plt.xlabel("Class")
plt.ylabel("Avg Heart Rate")
plt.show()


In [ ]:
df.groupby("target_class")["EEG_spike_rate"].mean().plot(kind="bar")

plt.title("EEG Spike Rate Across Conditions")
plt.xlabel("Class")
plt.ylabel("Avg EEG Spike Rate")
plt.show()


In [ ]:
df.groupby("target_class")["respiration_rate"].mean().plot(kind="bar")

plt.title("Respiration Rate Across Conditions")
plt.xlabel("Class")
plt.ylabel("Avg Respiration Rate")
plt.show()


In [ ]:
df.groupby("target_class")["SpO2_level"].mean().plot(kind="bar")

plt.title("Oxygen Saturation Across Conditions")
plt.xlabel("Class")
plt.ylabel("Avg SpO2 Level")
plt.show()


In [ ]:
cnn_model = Sequential([
    Conv1D(64, 2, activation="relu", input_shape=(8,1)),
    MaxPooling1D(2),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(4, activation="softmax")
])

cnn_model.compile(optimizer="adam",
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])

cnn_model.summary()


In [ ]:
history_cnn = cnn_model.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_test_seq, y_test_cat),
    epochs=50,
    batch_size=32,
    verbose=1
)


In [ ]:
lstm_model = Sequential([
    LSTM(64, return_sequences=False, input_shape=(8,1)),
    Dropout(0.3),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(4, activation="softmax")
])

lstm_model.compile(optimizer="adam",
                   loss="categorical_crossentropy",
                   metrics=["accuracy"])

lstm_model.summary()


In [ ]:
history_lstm = lstm_model.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_test_seq, y_test_cat),
    epochs=50,
    batch_size=32,
    verbose=1
)


In [ ]:
input_layer = Input(shape=(8,1))

# CNN Branch
cnn_branch = Conv1D(64, 2, activation="relu")(input_layer)
cnn_branch = MaxPooling1D(2)(cnn_branch)
cnn_branch = Flatten()(cnn_branch)

# LSTM Branch
lstm_branch = LSTM(64)(input_layer)

# Fusion
merged = Concatenate()([cnn_branch, lstm_branch])

dense = Dense(128, activation="relu")(merged)
drop = Dropout(0.3)(dense)

output = Dense(4, activation="softmax")(drop)

hybrid_model = Model(inputs=input_layer, outputs=output)

hybrid_model.compile(optimizer="adam",
                     loss="categorical_crossentropy",
                     metrics=["accuracy"])

hybrid_model.summary()


In [ ]:
history_hybrid = hybrid_model.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_test_seq, y_test_cat),
    epochs=25,
    batch_size=32,
    verbose=1
)


In [ ]:
cnn_acc = cnn_model.evaluate(X_test_seq, y_test_cat)[1]
lstm_acc = lstm_model.evaluate(X_test_seq, y_test_cat)[1]
hybrid_acc = hybrid_model.evaluate(X_test_seq, y_test_cat)[1]

print("\nCNN Accuracy:", cnn_acc)
print("LSTM Accuracy:", lstm_acc)
print("Hybrid Accuracy:", hybrid_acc)

# Save Best Model
best_acc = max(cnn_acc, lstm_acc, hybrid_acc)

if best_acc == cnn_acc:
    cnn_model.save("best_model.h5")
    best_name = "CNN"
elif best_acc == lstm_acc:
    lstm_model.save("best_model.h5")
    best_name = "LSTM"
else:
    hybrid_model.save("best_model.h5")
    best_name = "Hybrid CNN+LSTM"

print("\n Best Model Saved:", best_name)


In [ ]:
best_model = tf.keras.models.load_model("best_model.h5")

y_pred = best_model.predict(X_test_seq)
y_pred_classes = np.argmax(y_pred, axis=1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_classes))


In [ ]:
plt.plot(history_hybrid.history["accuracy"], label="Train Accuracy")
plt.plot(history_hybrid.history["val_accuracy"], label="Val Accuracy")

plt.title("Hybrid CNN+LSTM Learning Curve")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


In [ ]:
def predict_patient(sample_dict):

    input_df = pd.DataFrame([sample_dict])

    input_scaled = scaler.transform(input_df)

    input_seq = input_scaled.reshape(1, 8, 1)

    prediction = best_model.predict(input_seq)
    predicted_class = np.argmax(prediction)

    classes = {
        0: "Normal",
        1: "Epileptic Seizure",
        2: "Cardiac Abnormality",
        3: "Respiratory Distress"
    }

    print("Predicted Condition:", classes[predicted_class])
    print("Class Probabilities:", prediction)


In [ ]:
sample_patient = {
    "EEG_band_power": 78,
    "EEG_entropy": 1.1,
    "EEG_spike_rate": 6.5,
    "heart_rate": 115,
    "HRV": 40,
    "RR_interval": 0.65,
    "respiration_rate": 25,
    "SpO2_level": 92
}

predict_patient(sample_patient)


In [ ]:
sample1 = {
    "EEG_band_power": 58,
    "EEG_entropy": 0.85,
    "EEG_spike_rate": 2.0,
    "heart_rate": 78,
    "HRV": 60,
    "RR_interval": 0.82,
    "respiration_rate": 16,
    "SpO2_level": 98
}

predict_patient(sample1)


In [ ]:
sample2 = {
    "EEG_band_power": 90,
    "EEG_entropy": 1.35,
    "EEG_spike_rate": 8.8,
    "heart_rate": 125,
    "HRV": 35,
    "RR_interval": 0.55,
    "respiration_rate": 24,
    "SpO2_level": 93
}

predict_patient(sample2)


In [ ]:
sample3 = {
    "EEG_band_power": 62,
    "EEG_entropy": 0.95,
    "EEG_spike_rate": 3.0,
    "heart_rate": 45,
    "HRV": 22,
    "RR_interval": 1.20,
    "respiration_rate": 18,
    "SpO2_level": 96
}

predict_patient(sample3)


In [ ]:
sample4 = {
    "EEG_band_power": 60,
    "EEG_entropy": 0.90,
    "EEG_spike_rate": 2.5,
    "heart_rate": 95,
    "HRV": 50,
    "RR_interval": 0.75,
    "respiration_rate": 32,
    "SpO2_level": 86
}

predict_patient(sample4)


In [ ]:
samples = {
    "Normal Patient": normal_sample,
    "Seizure Patient": sample1,
    "Cardiac Patient": sample2,
    "Respiratory Patient": sample3
}

for name, sample in samples.items():
    print("\n==============================")
    print("Testing:", name)
    predict_patient(sample)


In [ ]:
custom_sample = {
    "EEG_band_power": np.random.uniform(55, 90),
    "EEG_entropy": np.random.uniform(0.8, 1.4),
    "EEG_spike_rate": np.random.uniform(1, 9),
    "heart_rate": np.random.uniform(40, 130),
    "HRV": np.random.uniform(20, 70),
    "RR_interval": np.random.uniform(0.5, 1.3),
    "respiration_rate": np.random.uniform(12, 35),
    "SpO2_level": np.random.uniform(85, 99)
}

predict_patient(custom_sample)


In [ ]:
manual_sample = {
    "EEG_band_power": float(input("Enter EEG Band Power: ")),
    "EEG_entropy": float(input("Enter EEG Entropy: ")),
    "EEG_spike_rate": float(input("Enter EEG Spike Rate: ")),
    "heart_rate": float(input("Enter Heart Rate: ")),
    "HRV": float(input("Enter HRV Value: ")),
    "RR_interval": float(input("Enter RR Interval: ")),
    "respiration_rate": float(input("Enter Respiration Rate: ")),
    "SpO2_level": float(input("Enter SpO2 Level: "))
}

predict_patient(manual_sample)


In [ ]:
def predict_with_probabilities(sample_dict):

    input_df = pd.DataFrame([sample_dict])
    input_scaled = scaler.transform(input_df)
    input_seq = input_scaled.reshape(1, 8, 1)

    prediction = best_model.predict(input_seq)[0]

    classes = ["Normal", "Seizure", "Cardiac", "Respiratory"]

    print("\nPrediction Probabilities:")
    for i, prob in enumerate(prediction):
        print(f"{classes[i]} : {prob*100:.2f}%")


In [ ]:
predict_with_probabilities(sample1)
